# GRU + Greedy

In [1]:
import random, time
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, random_split
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence 
from tqdm import tqdm

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE, EMB_DIM, HIDDEN_DIM, EPOCHS = 128, 128, 256, 30
MODEL_FILE = 'gru_greedy.pt'
print('Device:', DEVICE)

Device: cuda


In [2]:
def load_file(path):
    with open(path, encoding='utf-8') as f:
        return dict(line.rstrip('\n').split('\t', 1) for line in f if line.strip())

en, ko = load_file('num2txt.txt'), load_file('num2kor.txt')
assert en.keys() == ko.keys(), 'Number keys do not match.'
rows = [(num, en[num], ko[num]) for num in en]

train_rows, val_rows, test_rows = random_split(
    rows,
    [0.8, 0.1, 0.1],
    generator=torch.Generator().manual_seed(SEED)
)

LANGS = ('num', 'en', 'ko')
def tokenize(text, lang):
    return text.split() if lang == 'en' else list(text)

def make_pairs(data):
    pairs = []
    for num, english, korean in data:
        texts = {'num': num, 'en': english, 'ko': korean}
        for src in LANGS:
            for tgt in LANGS:
                if src != tgt:
                    pairs.append((src, tgt, texts[src], texts[tgt]))
    return pairs

train_pairs = make_pairs(train_rows)
val_pairs = make_pairs(val_rows)
test_pairs = make_pairs(test_rows)

special = ['<pad>', '<bos>', '<eos>', '<unk>'] + [f'<2{x}>' for x in LANGS]
tokens = set()
for src, tgt, a, b in train_pairs:
    tokens.update(tokenize(a, src))
    tokens.update(tokenize(b, tgt))
vocab = special + sorted(tokens - set(special))
stoi = {token: i for i, token in enumerate(vocab)}
itos = {i: token for token, i in stoi.items()}
PAD, BOS, EOS, UNK = (stoi[x] for x in special[:4])
print('Records:', len(rows), '| Vocabulary:', len(vocab), '| Training pairs:', len(train_pairs))

Records: 100000 | Vocabulary: 65 | Training pairs: 480000


In [3]:
def ids(text, lang):
    return [stoi.get(token, UNK) for token in tokenize(text, lang)]

class TranslationDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, index):
        src, tgt, a, b = self.pairs[index]
        x = [stoi[f'<2{tgt}>']] + ids(a, src) + [EOS]
        y = [BOS] + ids(b, tgt) + [EOS]
        return torch.tensor(x), torch.tensor(y)

def collate(batch):
    x, y = zip(*batch)
    return (pad_sequence(x, batch_first=True, padding_value=PAD),
            pad_sequence(y, batch_first=True, padding_value=PAD))

train_loader = DataLoader(TranslationDataset(train_pairs), batch_size=BATCH_SIZE,
                          shuffle=True, collate_fn=collate)
val_loader = DataLoader(TranslationDataset(val_pairs), batch_size=BATCH_SIZE,
                        shuffle=False, collate_fn=collate)

In [4]:
class Seq2Seq(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, EMB_DIM, padding_idx=PAD)
        self.encoder = nn.GRU(EMB_DIM, HIDDEN_DIM, batch_first=True)
        self.decoder = nn.GRU(EMB_DIM, HIDDEN_DIM, batch_first=True)
        self.fc = nn.Linear(HIDDEN_DIM, vocab_size)

    def encode(self, x):
        lengths = (x != PAD).sum(1).cpu()
        embedded = self.embedding(x)
        packed = pack_padded_sequence(embedded, lengths, batch_first=True,
                                      enforce_sorted=False)
        _, state = self.encoder(packed)
        return state

    def forward(self, x, y):
        state = self.encode(x)
        out, _ = self.decoder(self.embedding(y[:, :-1]), state)
        return self.fc(out)

model = Seq2Seq(len(vocab)).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss(ignore_index=PAD)
print('Parameters:', sum(p.numel() for p in model.parameters()))

Parameters: 617921


In [5]:
# def run_epoch(loader, training):
#     model.train(training)
#     total_loss, total_tokens = 0.0, 0
#     for x, y in tqdm(loader, leave=False):
#         x, y = x.to(DEVICE), y.to(DEVICE)
#         with torch.set_grad_enabled(training):
#             logits = model(x, y)
#             target = y[:, 1:]
#             loss = criterion(logits.reshape(-1, len(vocab)), target.reshape(-1))
#             if training:
#                 optimizer.zero_grad()
#                 loss.backward()
#                 nn.utils.clip_grad_norm_(model.parameters(), 1.0)
#                 optimizer.step()
#         count = (target != PAD).sum().item()
#         total_loss += loss.item() * count
#         total_tokens += count
#     return total_loss / total_tokens

# best_loss = float('inf')
# for epoch in range(1, EPOCHS + 1):
#     start = time.time()
#     train_loss = run_epoch(train_loader, True)
#     val_loss = run_epoch(val_loader, False)
#     print(f'Epoch {epoch:02d} | train {train_loss:.4f} | val {val_loss:.4f} | {time.time()-start:.0f}s')
#     if val_loss < best_loss:
#         best_loss = val_loss
#         torch.save({'model': model.state_dict(), 'vocab': vocab,
#                     'emb_dim': EMB_DIM, 'hidden_dim': HIDDEN_DIM}, MODEL_FILE)
# print('Best validation loss:', round(best_loss, 4))

In [6]:
@torch.no_grad()
def translate(text, source_lang, target_lang, net=model, max_len=60):
    net.eval()
    x = [stoi[f'<2{target_lang}>']] + ids(str(text), source_lang) + [EOS]
    x = torch.tensor([x], device=DEVICE)
    state = net.encode(x)
    token = torch.tensor([[BOS]], device=DEVICE)
    output = []
    for _ in range(max_len):
        hidden, state = net.decoder(net.embedding(token), state)
        token = net.fc(hidden).argmax(-1)  # greedy: 매 단계 가장 높은 점수 토큰 선택
        token_id = token.item()
        if token_id == EOS:
            break
        output.append(itos[token_id])
    return ' '.join(output) if target_lang == 'en' else ''.join(output)

model.load_state_dict(
    torch.load(MODEL_FILE, map_location=DEVICE, weights_only=True)['model']
)

for src, tgt, text, gt in test_pairs[:6]:
    predicted = translate(text, src, tgt)
    correct = predicted == gt  # 예측 결과와 gt가 같으면 PASS만 표시, 다르면 결과 표시
    print(f'{src}->{tgt} | {"PASS" if correct else "FAIL"}')
    if not correct:
        print(f'  predicted: {predicted}')
        print(f'  gt:  {gt}')

num->en | PASS
num->ko | PASS
en->num | PASS
en->ko | PASS
ko->num | PASS
ko->en | PASS


In [7]:
counts = {(a, b): [0, 0] for a in LANGS for b in LANGS if a != b}
for src, tgt, text, gt in tqdm(test_pairs):
    predicted = translate(text, src, tgt)
    counts[(src, tgt)][0] += (predicted == gt)
    counts[(src, tgt)][1] += 1
for direction, (correct, total) in counts.items():
    print(f'{direction[0]} -> {direction[1]}: {100 * correct / total:.2f}%')

100%|██████████| 60000/60000 [07:58<00:00, 125.39it/s]

num -> en: 99.17%
num -> ko: 99.18%
en -> num: 99.76%
en -> ko: 99.52%
ko -> num: 99.82%
ko -> en: 99.57%


## 사용방법
- torch는 운영환경에 맞게 따로 설치 후, ```pip install tqdm notebook```로 필요한 라이브러리를 설치해주세요.
- 실행 전 이 파일과 동일한 경로에 .pt 파일과 .txt 파일이 있는지 확인하세요.
- 트레이닝 코드(5번째 셀)만 주석처리했습니다. 주석을 유지한 채로 전체 실행 시 마지막 셀에서 테스트 결과를 확인할 수 있습니다.
- 직접 모델 트레이닝 시 첫 번째 셀의 MODEL_FILE 값을 변경 후 run_epoch 함수가 작성된 5번째 셀의 주석을 해제 후 실행해주세요.